# Lab 2.2 — Structured Portfolio Extraction with Pydantic

## Introduction

This is the **second lab** in the AI Models module, building on [Lab 2.1](2_lab2.1.ipynb). Where Lab 2.1 compared free-text responses across providers, this lab focuses on getting **reliable structured data** back from an LLM.

You will build a portfolio extraction pipeline that takes **noisy OCR text** from a DIME mobile investment app screenshot and converts it into typed Python objects. Real-world documents — especially OCR output — are messy: garbled tickers, stray UI labels, broken line breaks, and currency symbols mixed into numbers. Instead of asking the LLM for JSON and parsing it yourself, you will use **Pydantic models** and OpenAI's **structured outputs** API to enforce a schema at generation time.

The sample input lives in `portfolio/my-port.md`. It simulates OCR text captured from a portfolio screen. Your goal is to extract each stock holding with fields such as ticker, USD/THB values, profit/loss, and allocation percentage.

Before starting, make sure you have:
- Completed Lab 2.1 and the Setup labs
- `OPENAI_API_KEY` set in your `.env` file
- The sample file `portfolio/my-port.md` in place (or replace it with your own OCR text)

---

## Intention (Learning Objectives)

By the end of this lab, you should be able to:

1. **Define Pydantic models** — describe the exact shape of structured data you expect from an LLM
2. **Write an extraction system prompt** — instruct the model to ignore OCR noise and extract specific financial fields
3. **Use OpenAI structured outputs** — call `openai.beta.chat.completions.parse()` with a Pydantic `response_format`
4. **Load raw input from a file** — read OCR text and pass it as the user message
5. **Return typed Python objects** — access parsed holdings as a list of `MyPortfolioModel` instances, not raw strings

---

## Lab Structure

| Part | Content |
|------|---------|
| **Part 1** | Imports |
| **Part 2** | Define `MyPortfolioModel` and `PortfolioExtraction` Pydantic schemas |
| **Part 3** | Load environment variables and create the OpenAI client |
| **Part 4** | Write the extraction system prompt |
| **Part 5** | Load sample OCR text from `portfolio/my-port.md` |
| **Part 6** | Implement `extract_portfolio()` and run extraction |

> **Note:** Run cells top to bottom in order.

## Part 1 — Setup

Import the packages we need: `python-dotenv` for environment variables, `OpenAI` for the API client, and `Pydantic` for structured data models.

In [11]:
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
import os

## Part 2 — Define Pydantic Models

Describe the schema for a single holding (`MyPortfolioModel`) and a wrapper that holds a list of holdings (`PortfolioExtraction`). OpenAI will use these models to constrain its output.

In [12]:
class MyPortfolioModel(BaseModel):
    underlying: str
    holding_value_usd: float
    holding_value_thb: float
    profit_amount_usd: float
    profit_percentage: float
    allocation_percentage: float


class PortfolioExtraction(BaseModel):
    holdings: list[MyPortfolioModel]

## Part 3 — Load Environment & Create Client

Load your `.env` file and create the OpenAI client.

In [13]:
load_dotenv(override=True)
openai = OpenAI()

## Part 4 — Write the Extraction System Prompt

The system prompt tells the LLM its role (financial data extraction specialist), what fields to extract, and how to handle noisy OCR text. Read through it carefully — prompt quality directly affects extraction accuracy.

In [14]:
extract_portfolio_system_prompt = """You are a financial data extraction specialist. \
Your role is to parse noisy OCR text from a DIME mobile investment app portfolio screenshot submitted by an applicant, \
and convert it into accurate structured portfolio data.

## Intention
The applicant captured their stock portfolio from the DIME app. The OCR output is messy—it may include garbled tickers, \
stray UI labels (e.g. "Sort by", "Home", "Invest", "Assets"), copyright symbols, and broken line breaks. \
Your job is to identify each real stock holding and extract its financial fields reliably, ignoring all non-holding text.

## Fields to extract (per holding)
- underlying: the stock ticker or underlying asset symbol (e.g. S, UNH, QUBT, MSFT, META, HIG). Infer the correct ticker when OCR corrupts the symbol.
- holding_value_usd: current holding value in US dollars
- holding_value_thb: current holding value in Thai baht (THB)
- profit_amount_usd: unrealized profit or loss in USD (negative for losses)
- profit_percentage: unrealized profit or loss as a percentage (negative for losses)
- allocation_percentage: this holding's share of total portfolio value as a percentage

## Rules
- Extract every distinct stock holding visible in the text; do not skip rows because of OCR noise.
- Parse numbers carefully: remove currency labels, thousand separators, and OCR artifacts before converting to floats.
- Preserve signs on profit/loss values (e.g. -22.02%, -42.15 USD).
- Do not invent holdings or values that are not supported by the OCR text.
- If one field is unreadable, infer it only when strongly implied by nearby values on the same row.
- Return structured data only—no explanations or commentary."""

## Part 5 — Load Sample OCR Text

Read the sample portfolio OCR text from `portfolio/my-port.md`. You can open this file to see the raw input — notice the UI labels, copyright symbols, and garbled characters the model must ignore.

In [16]:
with open("portfolio/my-port.md", "r", encoding="utf-8") as f:
    my_portfolio_text = f.read()

## Part 6 — Extract Portfolio & Run

Implement `extract_portfolio()` using `openai.beta.chat.completions.parse()`. Pass `PortfolioExtraction` as the `response_format` so the API returns validated Pydantic objects. Then run it on the sample text and inspect the results.

In [ ]:
def extract_portfolio(ocr_text) -> list[MyPortfolioModel]:
    messages = [
        {"role": "system", "content": extract_portfolio_system_prompt},
        {"role": "user", "content": "Extract the portfolio from the text: " + ocr_text},
    ]
    
    response = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=messages,
        response_format=PortfolioExtraction,
    )
    return response.choices[0].message.parsed.holdings

In [24]:
extract_portfolio(my_portfolio_text)

[MyPortfolioModel(underlying='S', holding_value_usd=303.09, holding_value_thb=9920.17, profit_amount_usd=48.71, profit_percentage=19.15, allocation_percentage=24.12),
 MyPortfolioModel(underlying='UNH', holding_value_usd=276.17, holding_value_thb=9038.95, profit_amount_usd=54.31, profit_percentage=24.48, allocation_percentage=21.98),
 MyPortfolioModel(underlying='QUBT', holding_value_usd=149.27, holding_value_thb=4885.58, profit_amount_usd=-42.15, profit_percentage=-22.02, allocation_percentage=11.88),
 MyPortfolioModel(underlying='MSFT', holding_value_usd=128.32, holding_value_thb=4199.9, profit_amount_usd=-19.76, profit_percentage=-13.34, allocation_percentage=10.21),
 MyPortfolioModel(underlying='META', holding_value_usd=118.37, holding_value_thb=3874.28, profit_amount_usd=-21.73, profit_percentage=15.51, allocation_percentage=9.42),
 MyPortfolioModel(underlying='HIG', holding_value_usd=75.21, holding_value_thb=2461.63, profit_amount_usd=-1.24, profit_percentage=-1.62, allocation_pe

---

## Next Steps

Compare the extracted holdings against the raw OCR text in `portfolio/my-port.md`. Try replacing the file with your own OCR output, adding fields to `MyPortfolioModel`, or printing individual holdings in a formatted table.